In [1]:
# Load the cleaned hospital datasets from data/processed
# so that all EDA is performed on validated and cleaned data.
import pandas as pd
from pathlib import Path

processed_base = Path("../data/processed")

admissions = pd.read_csv(
    processed_base / "admissions/admissions_clean.csv",
    parse_dates=["admission_date", "discharge_date"]
)

appointments = pd.read_csv(
    processed_base / "appointments/appointments_clean.csv",
    parse_dates=["appointment_date", "booking_date"]
)

beds = pd.read_csv(
    processed_base / "beds/beds_clean.csv",
    parse_dates=["record_date"]
)

departments = pd.read_csv(
    processed_base / "departments/departments_clean.csv"
)

equipment = pd.read_csv(
    processed_base / "equipment/equipment_clean.csv",
    parse_dates=[
        "installation_date",
        "last_maintenance_date",
        "next_maintenance_date"
    ]
)

print("All cleaned datasets loaded successfully!")


All cleaned datasets loaded successfully!


In [2]:
# Display the size and column names of each cleaned dataset
# to understand the structure before performing detailed analysis.
datasets = {
    "Admissions": admissions,
    "Appointments": appointments,
    "Beds": beds,
    "Departments": departments,
    "Equipment": equipment
}

for name, df in datasets.items():
    print("\n==============================")
    print(name)
    print("==============================")
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Column names:", df.columns.tolist())


Admissions
Rows: 50000
Columns: 11
Column names: ['admission_id', 'patient_id', 'department_id', 'admission_date', 'discharge_date', 'admission_type', 'age', 'gender', 'length_of_stay', 'bed_type', 'discharge_status']

Appointments
Rows: 60000
Columns: 8
Column names: ['appointment_id', 'patient_id', 'department_id', 'appointment_date', 'appointment_type', 'booking_date', 'wait_time_days', 'appointment_status']

Beds
Rows: 15344
Columns: 8
Column names: ['bed_record_id', 'department_id', 'record_date', 'bed_type', 'total_beds', 'occupied_beds', 'available_beds', 'occupancy_rate']

Departments
Rows: 14
Columns: 4
Column names: ['department_id', 'department_name', 'department_type', 'hospital_area']

Equipment
Rows: 5000
Columns: 10
Column names: ['equipment_id', 'department_id', 'equipment_type', 'installation_date', 'last_maintenance_date', 'next_maintenance_date', 'equipment_status', 'usage_hours', 'maintenance_count', 'failure_count']


In [3]:
# Generate summary statistics for numerical columns
# to understand the distribution and range of important hospital metrics.
print("=== Admissions ===")
print(admissions.describe())

print("\n=== Appointments ===")
print(appointments.describe())

print("\n=== Beds ===")
print(beds.describe())

print("\n=== Equipment ===")
print(equipment.describe())

=== Admissions ===
                   admission_date                 discharge_date  \
count                       50000                          50000   
mean   2024-07-02 01:07:09.696000  2024-07-05 20:45:13.536000256   
min           2023-01-01 00:00:00            2023-01-01 00:00:00   
25%           2023-10-17 00:00:00            2023-10-21 00:00:00   
50%           2024-06-28 00:00:00            2024-07-02 00:00:00   
75%           2025-03-22 00:00:00            2025-03-25 06:00:00   
max           2025-12-31 00:00:00            2026-01-27 00:00:00   
std                           NaN                            NaN   

                age  length_of_stay  
count  50000.000000    50000.000000  
mean      44.750400        3.818100  
min        0.000000        0.000000  
25%       29.000000        2.000000  
50%       46.000000        3.000000  
75%       61.000000        5.000000  
max      100.000000       44.000000  
std       22.054655        3.152093  

=== Appointments ===
    

In [4]:
# Count the records in important categorical columns
# to understand the distribution of hospital operations.
print("=== Admissions ===")

print("\nAdmission Type:")
print(admissions["admission_type"].value_counts())

print("\nGender:")
print(admissions["gender"].value_counts())

print("\nBed Type:")
print(admissions["bed_type"].value_counts())

print("\nDischarge Status:")
print(admissions["discharge_status"].value_counts())

print("\n=== Appointments ===")

print("\nAppointment Type:")
print(appointments["appointment_type"].value_counts())

print("\nAppointment Status:")
print(appointments["appointment_status"].value_counts())

print("\n=== Equipment ===")

print("\nEquipment Status:")
print(equipment["equipment_status"].value_counts())

=== Admissions ===

Admission Type:
admission_type
Emergency    19805
Elective     18204
Urgent       11991
Name: count, dtype: int64

Gender:
gender
Female     26291
Male       21862
Other       1347
Unknown      500
Name: count, dtype: int64

Bed Type:
bed_type
General         23257
Semi-Private     9579
ICU              9467
Private          7297
Unknown           400
Name: count, dtype: int64

Discharge Status:
discharge_status
Discharged     41007
Transferred     3545
Referred        3178
Deceased        1270
Unknown         1000
Name: count, dtype: int64

=== Appointments ===

Appointment Type:
appointment_type
Follow-up           22592
New Consultation    18897
Diagnostic           9997
Procedure            7914
Unknown               600
Name: count, dtype: int64

Appointment Status:
appointment_status
Completed      47460
Cancelled       4577
No-Show         4308
Rescheduled     2755
Unknown          900
Name: count, dtype: int64

=== Equipment ===

Equipment Status:
equipment_

In [5]:
# Calculate total admissions and average length of stay for each department
# to identify departments with high patient volume and longer stays.
department_admissions = (
    admissions.groupby("department_id")
    .agg(
        total_admissions=("admission_id", "count"),
        average_length_of_stay=("length_of_stay", "mean")
    )
    .reset_index()
)

department_admissions["average_length_of_stay"] = (
    department_admissions["average_length_of_stay"].round(2)
)

print(department_admissions)

   department_id  total_admissions  average_length_of_stay
0            D01              9076                    2.78
1            D02              4479                    4.82
2            D03              3423                    3.28
3            D04              3066                    3.31
4            D05              4002                    3.35
5            D06              7091                    3.32
6            D07              4408                    3.33
7            D08              2975                    7.09
8            D09              3989                    4.46
9            D10              2466                    6.35
10           D11              1520                    3.27
11           D12              1047                    3.21
12           D13              1456                    3.29
13           D14              1002                    3.34


In [6]:
# Join department names with admission statistics
# so the analysis is understandable to hospital administrators.
department_admissions_named = department_admissions.merge(
    departments[["department_id", "department_name", "department_type"]],
    on="department_id",
    how="left"
)

department_admissions_named = department_admissions_named[
    [
        "department_id",
        "department_name",
        "department_type",
        "total_admissions",
        "average_length_of_stay"
    ]
]

print(department_admissions_named)

   department_id   department_name department_type  total_admissions  \
0            D01         Emergency      Acute Care              9076   
1            D02        Cardiology  Specialty Care              4479   
2            D03       Orthopedics  Specialty Care              3423   
3            D04         Neurology  Specialty Care              3066   
4            D05        Pediatrics    General Care              4002   
5            D06  General Medicine    General Care              7091   
6            D07   General Surgery   Surgical Care              4408   
7            D08               ICU   Critical Care              2975   
8            D09         Maternity    General Care              3989   
9            D10          Oncology  Specialty Care              2466   
10           D11         Radiology      Diagnostic              1520   
11           D12       Dermatology  Specialty Care              1047   
12           D13               ENT  Specialty Care              

In [7]:
# Calculate the number of admissions for each month
# to identify demand trends and seasonal patterns over time.
monthly_admissions = (
    admissions
    .groupby(admissions["admission_date"].dt.to_period("M"))
    .size()
    .reset_index(name="total_admissions")
)

monthly_admissions["admission_date"] = (
    monthly_admissions["admission_date"].astype(str)
)

print(monthly_admissions)

   admission_date  total_admissions
0         2023-01              1601
1         2023-02              1425
2         2023-03              1566
3         2023-04              1420
4         2023-05              1321
5         2023-06              1133
6         2023-07               996
7         2023-08              1057
8         2023-09              1176
9         2023-10              1568
10        2023-11              1594
11        2023-12              1583
12        2024-01              1658
13        2024-02              1424
14        2024-03              1652
15        2024-04              1490
16        2024-05              1322
17        2024-06              1106
18        2024-07              1005
19        2024-08              1104
20        2024-09              1290
21        2024-10              1497
22        2024-11              1683
23        2024-12              1676
24        2025-01              1621
25        2025-02              1442
26        2025-03           

In [ ]:
# Analyze appointment outcomes to measure demand, cancellations,
# no-shows, and rescheduling patterns.
appointment_status = (
    appointments["appointment_status"]
    .value_counts()
    .reset_index()
)

appointment_status.columns = ["appointment_status", "total_appointments"]

appointment_status["percentage"] = (
    appointment_status["total_appointments"]
    / len(appointments) * 100
).round(2)

print(appointment_status)

  appointment_status  total_appointments  percentage
0          Completed               47460       79.10
1          Cancelled                4577        7.63
2            No-Show                4308        7.18
3        Rescheduled                2755        4.59
4            Unknown                 900        1.50


In [9]:
# Calculate average appointment waiting time for each department
# to identify departments where patients experience longer waiting periods.
department_wait_time = (
    appointments.groupby("department_id")
    .agg(
        total_appointments=("appointment_id", "count"),
        average_wait_time=("wait_time_days", "mean")
    )
    .reset_index()
)

department_wait_time["average_wait_time"] = (
    department_wait_time["average_wait_time"].round(2)
)

department_wait_time = department_wait_time.merge(
    departments[["department_id", "department_name"]],
    on="department_id",
    how="left"
)

department_wait_time = department_wait_time[
    [
        "department_id",
        "department_name",
        "total_appointments",
        "average_wait_time"
    ]
].sort_values("average_wait_time", ascending=False)

print(department_wait_time)

   department_id   department_name  total_appointments  average_wait_time
13           D14        Nephrology                1213              13.19
11           D12       Dermatology                1135              12.86
10           D11         Radiology                1845              12.05
12           D13               ENT                1802              11.60
3            D04         Neurology                3614              10.59
8            D09         Maternity                4746              10.37
6            D07   General Surgery                5390              10.16
2            D03       Orthopedics                4147              10.15
1            D02        Cardiology                5472              10.14
7            D08               ICU                3617              10.12
4            D05        Pediatrics                4901              10.06
5            D06  General Medicine                8368               9.86
0            D01         Emergency    

In [ ]:
# Calculate appointment no-show rate for each department
# to identify departments with higher rates of missed appointments.

department_noshow = (
    appointments.assign(
        is_no_show=(
            appointments["appointment_status"] == "No-Show"
        ).astype(int)
    )
    .groupby("department_id")
    .agg(
        total_appointments=("appointment_id", "count"),
        no_show_count=("is_no_show", "sum")
    )
    .reset_index()
)

department_noshow["no_show_rate"] = (
    department_noshow["no_show_count"]
    / department_noshow["total_appointments"]
    * 100
).round(2)

department_noshow = department_noshow.merge(
    departments[["department_id", "department_name"]],
    on="department_id",
    how="left"
)

department_noshow = department_noshow[
    [
        "department_id",
        "department_name",
        "total_appointments",
        "no_show_count",
        "no_show_rate"
    ]
].sort_values("no_show_rate", ascending=False)

print(department_noshow)


   department_id   department_name  total_appointments  no_show_count  \
11           D12       Dermatology                1135             95   
13           D14        Nephrology                1213             95   
12           D13               ENT                1802            140   
9            D10          Oncology                3085            238   
4            D05        Pediatrics                4901            377   
7            D08               ICU                3617            278   
10           D11         Radiology                1845            141   
1            D02        Cardiology                5472            401   
3            D04         Neurology                3614            265   
2            D03       Orthopedics                4147            295   
0            D01         Emergency               10665            751   
6            D07   General Surgery                5390            366   
5            D06  General Medicine                8

In [11]:
# Display the complete department no-show analysis
# in a compact format so no rows are truncated.

print(
    department_noshow[
        [
            "department_name",
            "total_appointments",
            "no_show_count",
            "no_show_rate"
        ]
    ].to_string(index=False)
)

 department_name  total_appointments  no_show_count  no_show_rate
     Dermatology                1135             95          8.37
      Nephrology                1213             95          7.83
             ENT                1802            140          7.77
        Oncology                3085            238          7.71
      Pediatrics                4901            377          7.69
             ICU                3617            278          7.69
       Radiology                1845            141          7.64
      Cardiology                5472            401          7.33
       Neurology                3614            265          7.33
     Orthopedics                4147            295          7.11
       Emergency               10665            751          7.04
 General Surgery                5390            366          6.79
General Medicine                8368            561          6.70
       Maternity                4746            305          6.43


department_id  department_name  average_occupancy_rate  average_occupied_beds  average_available_beds
          D08              ICU                 8490.62                  16.95                    3.01
          D01        Emergency                 7764.80                  31.03                    8.93
          D06 General Medicine                 7040.56                  31.65                   13.31
          D09        Maternity                 7034.42                  16.85                    7.10
          D10         Oncology                 7031.05                  15.44                    6.52
          D02       Cardiology                 5744.14                  17.21                   12.75
          D11        Radiology                 5724.72                   5.70                    4.26
          D13              ENT                 5717.59                   5.70                    4.27
          D05       Pediatrics                 5716.10                  14.84     

In [15]:
# Calculate average bed occupancy for each department
# occupancy_rate is already stored as a percentage (for example, 75.0 = 75%).

department_occupancy = (
    beds.groupby("department_id")
    .agg(
        average_occupancy_rate=("occupancy_rate", "mean"),
        average_occupied_beds=("occupied_beds", "mean"),
        average_available_beds=("available_beds", "mean")
    )
    .reset_index()
)

department_occupancy["average_occupancy_rate"] = (
    department_occupancy["average_occupancy_rate"].round(2)
)

department_occupancy["average_occupied_beds"] = (
    department_occupancy["average_occupied_beds"].round(2)
)

department_occupancy["average_available_beds"] = (
    department_occupancy["average_available_beds"].round(2)
)

department_occupancy = department_occupancy.merge(
    departments[["department_id", "department_name"]],
    on="department_id",
    how="left"
)

department_occupancy = department_occupancy[
    [
        "department_id",
        "department_name",
        "average_occupancy_rate",
        "average_occupied_beds",
        "average_available_beds"
    ]
].sort_values(
    "average_occupancy_rate",
    ascending=False
)

print(department_occupancy.to_string(index=False))

department_id  department_name  average_occupancy_rate  average_occupied_beds  average_available_beds
          D08              ICU                   84.91                  16.95                    3.01
          D01        Emergency                   77.65                  31.03                    8.93
          D06 General Medicine                   70.41                  31.65                   13.31
          D09        Maternity                   70.34                  16.85                    7.10
          D10         Oncology                   70.31                  15.44                    6.52
          D02       Cardiology                   57.44                  17.21                   12.75
          D11        Radiology                   57.25                   5.70                    4.26
          D13              ENT                   57.18                   5.70                    4.27
          D05       Pediatrics                   57.16                  14.84     

In [14]:
# Check the actual occupancy_rate values in the beds dataset
# to determine whether they are stored as decimals (0.85)
# or percentages (85.0).

print("First 10 occupancy rates:")
print(beds["occupancy_rate"].head(10))

print("\nMinimum:", beds["occupancy_rate"].min())
print("Maximum:", beds["occupancy_rate"].max())

First 10 occupancy rates:
0    75.0
1    77.5
2    85.0
3    85.0
4    80.0
5    87.5
6    87.5
7    92.5
8    85.0
9    77.5
Name: occupancy_rate, dtype: float64

Minimum: 25.0
Maximum: 100.0


In [16]:
# Calculate average bed occupancy by month
# to identify seasonal capacity trends and periods of higher utilization.

monthly_occupancy = (
    beds.groupby(
        beds["record_date"].dt.to_period("M")
    )
    .agg(
        average_occupancy_rate=("occupancy_rate", "mean")
    )
    .reset_index()
)

monthly_occupancy["record_date"] = (
    monthly_occupancy["record_date"].astype(str)
)

monthly_occupancy["average_occupancy_rate"] = (
    monthly_occupancy["average_occupancy_rate"].round(2)
)

print(monthly_occupancy.to_string(index=False))

record_date  average_occupancy_rate
    2023-01                   68.93
    2023-02                   67.88
    2023-03                   67.89
    2023-04                   65.71
    2023-05                   61.57
    2023-06                   56.08
    2023-07                   52.45
    2023-08                   54.21
    2023-09                   59.60
    2023-10                   66.28
    2023-11                   70.01
    2023-12                   69.97
    2024-01                   69.09
    2024-02                   68.39
    2024-03                   67.96
    2024-04                   66.60
    2024-05                   61.97
    2024-06                   55.81
    2024-07                   52.00
    2024-08                   54.12
    2024-09                   59.98
    2024-10                   66.46
    2024-11                   69.85
    2024-12                   70.07
    2025-01                   69.12
    2025-02                   68.09
    2025-03                 

In [17]:
# Analyze equipment status
# to understand how many hospital equipment units are operational,
# under maintenance, or out of service.

equipment_status = (
    equipment["equipment_status"]
    .value_counts()
    .reset_index()
)

equipment_status.columns = [
    "equipment_status",
    "total_equipment"
]

equipment_status["percentage"] = (
    equipment_status["total_equipment"]
    / len(equipment)
    * 100
).round(2)

print(equipment_status.to_string(index=False))

 equipment_status  total_equipment  percentage
      Operational             4009       80.18
Under Maintenance              726       14.52
   Out of Service              265        5.30


In [18]:
# Analyze equipment failures and maintenance activity
# to identify equipment types or departments with higher failure counts.

equipment_analysis = (
    equipment.groupby("equipment_type")
    .agg(
        total_equipment=("equipment_id", "count"),
        total_failures=("failure_count", "sum"),
        total_maintenance=("maintenance_count", "sum"),
        average_usage_hours=("usage_hours", "mean")
    )
    .reset_index()
)

equipment_analysis["failure_rate"] = (
    equipment_analysis["total_failures"]
    / equipment_analysis["total_equipment"]
).round(2)

equipment_analysis["average_usage_hours"] = (
    equipment_analysis["average_usage_hours"].round(2)
)

equipment_analysis = equipment_analysis.sort_values(
    "total_failures",
    ascending=False
)

print(equipment_analysis.to_string(index=False))

    equipment_type  total_equipment  total_failures  total_maintenance  average_usage_hours  failure_rate
     Infusion Pump              784            1034              14502             11408.97          1.32
   Patient Monitor              739             927              13692             11134.51          1.25
         Nebulizer              513             729               9610             11488.07          1.42
       ECG Machine              506             704               9539             11245.95          1.39
        Ventilator              375             520               7171             11575.66          1.39
     Defibrillator              379             464               7063             11487.67          1.22
     X-Ray Machine              328             428               5991             10778.82          1.30
    Surgical Table              260             355               5126             11816.80          1.37
Anesthesia Machine              241           

In [19]:
# Check whether equipment usage hours are related to failure counts
# to determine whether usage could be a useful ML feature.

usage_failure = (
    equipment.groupby("equipment_type")
    .agg(
        average_usage_hours=("usage_hours", "mean"),
        average_failures=("failure_count", "mean")
    )
    .reset_index()
)

usage_failure["average_usage_hours"] = (
    usage_failure["average_usage_hours"].round(2)
)

usage_failure["average_failures"] = (
    usage_failure["average_failures"].round(2)
)

print(usage_failure.to_string(index=False))

print("\nCorrelation between usage hours and failures:")
print(
    equipment["usage_hours"].corr(
        equipment["failure_count"]
    )
)

    equipment_type  average_usage_hours  average_failures
Anesthesia Machine             11257.91              1.43
        CT Scanner             11602.42              1.20
     Defibrillator             11487.67              1.22
  Dialysis Machine             11972.41              1.48
       ECG Machine             11245.95              1.39
         Incubator             10649.56              1.27
     Infusion Pump             11408.97              1.32
       MRI Scanner             10803.01              1.33
         Nebulizer             11488.07              1.42
   Patient Monitor             11134.51              1.25
    Surgical Table             11816.80              1.37
Ultrasound Machine             10431.19              1.20
        Ventilator             11575.66              1.39
     X-Ray Machine             10778.82              1.30

Correlation between usage hours and failures:
0.5522874738124501


In [20]:
# Analyze the relationship between maintenance activity and equipment failures
# to determine whether maintenance_count provides useful information
# for future equipment failure prediction.

maintenance_failure = (
    equipment.groupby("equipment_type")
    .agg(
        average_maintenance=("maintenance_count", "mean"),
        average_failures=("failure_count", "mean")
    )
    .reset_index()
)

maintenance_failure["average_maintenance"] = (
    maintenance_failure["average_maintenance"].round(2)
)

maintenance_failure["average_failures"] = (
    maintenance_failure["average_failures"].round(2)
)

print(maintenance_failure.to_string(index=False))

print("\nCorrelation between maintenance count and failures:")
print(
    equipment["maintenance_count"].corr(
        equipment["failure_count"]
    )
)

    equipment_type  average_maintenance  average_failures
Anesthesia Machine                19.03              1.43
        CT Scanner                19.41              1.20
     Defibrillator                18.64              1.22
  Dialysis Machine                19.29              1.48
       ECG Machine                18.85              1.39
         Incubator                17.97              1.27
     Infusion Pump                18.50              1.32
       MRI Scanner                18.51              1.33
         Nebulizer                18.73              1.42
   Patient Monitor                18.53              1.25
    Surgical Table                19.72              1.37
Ultrasound Machine                17.54              1.20
        Ventilator                19.12              1.39
     X-Ray Machine                18.27              1.30

Correlation between maintenance count and failures:
0.6826667572185887


In [21]:

# Combine admission and appointment activity by department
# to compare patient demand across the two operational datasets.

admission_summary = (
    admissions.groupby("department_id")
    .agg(
        total_admissions=("admission_id", "count")
    )
    .reset_index()
)

appointment_summary = (
    appointments.groupby("department_id")
    .agg(
        total_appointments=("appointment_id", "count")
    )
    .reset_index()
)

department_activity = admission_summary.merge(
    appointment_summary,
    on="department_id",
    how="outer"
)

department_activity = department_activity.merge(
    departments[["department_id", "department_name"]],
    on="department_id",
    how="left"
)

department_activity = department_activity[
    [
        "department_id",
        "department_name",
        "total_admissions",
        "total_appointments"
    ]
].sort_values(
    "total_admissions",
    ascending=False
)

print(department_activity.to_string(index=False))

department_id  department_name  total_admissions  total_appointments
          D01        Emergency              9076               10665
          D06 General Medicine              7091                8368
          D02       Cardiology              4479                5472
          D07  General Surgery              4408                5390
          D05       Pediatrics              4002                4901
          D09        Maternity              3989                4746
          D03      Orthopedics              3423                4147
          D04        Neurology              3066                3614
          D08              ICU              2975                3617
          D10         Oncology              2466                3085
          D11        Radiology              1520                1845
          D13              ENT              1456                1802
          D12      Dermatology              1047                1135
          D14       Nephrology    

In [22]:
# Check the distribution of equipment failure counts
# to determine whether failure prediction is suitable for ML.

print("Equipment failure statistics:")
print(equipment["failure_count"].describe())

print("\nFailure count distribution:")
print(equipment["failure_count"].value_counts().sort_index())

Equipment failure statistics:
count    5000.000000
mean        1.327800
std         1.588223
min         0.000000
25%         0.000000
50%         1.000000
75%         2.000000
max        12.000000
Name: failure_count, dtype: float64

Failure count distribution:
failure_count
0     2067
1     1189
2      753
3      457
4      288
5      133
6       65
7       29
8       12
9        5
10       1
12       1
Name: count, dtype: int64


In [23]:
# Check the appointment status distribution
# to determine whether no-show prediction is suitable for classification.

print("Appointment status distribution:")
print(appointments["appointment_status"].value_counts())

print("\nAppointment status percentages:")
print(
    appointments["appointment_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Appointment status distribution:
appointment_status
Completed      47460
Cancelled       4577
No-Show         4308
Rescheduled     2755
Unknown          900
Name: count, dtype: int64

Appointment status percentages:
appointment_status
Completed      79.10
Cancelled       7.63
No-Show         7.18
Rescheduled     4.59
Unknown         1.50
Name: proportion, dtype: float64


In [24]:
# Check the distribution of length of stay
# to determine whether length-of-stay prediction is suitable for regression.

print("Length of stay statistics:")
print(admissions["length_of_stay"].describe())

print("\nLength of stay distribution:")
print(
    admissions["length_of_stay"]
    .value_counts()
    .sort_index()
)

Length of stay statistics:
count    50000.000000
mean         3.818100
std          3.152093
min          0.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         44.000000
Name: length_of_stay, dtype: float64

Length of stay distribution:
length_of_stay
0     1913
1     8951
2     9825
3     8075
4     6175
5     4447
6     3077
7     2149
8     1559
9     1094
10     762
11     505
12     369
13     256
14     222
15     152
16     113
17      81
18      66
19      50
20      39
21      30
22      18
23      17
24      12
25       9
26      12
27       6
28       4
29       4
30       2
31       1
32       2
33       1
35       1
44       1
Name: count, dtype: int64


In [25]:
# Analyze daily occupied beds over time
# to determine whether the dataset supports bed-demand forecasting.

daily_bed_demand = (
    beds.groupby("record_date")
    .agg(
        total_occupied_beds=("occupied_beds", "sum"),
        total_available_beds=("available_beds", "sum")
    )
    .reset_index()
)

print("Daily bed-demand statistics:")
print(daily_bed_demand[
    ["total_occupied_beds", "total_available_beds"]
].describe())

print("\nDate range:")
print(
    daily_bed_demand["record_date"].min(),
    "to",
    daily_bed_demand["record_date"].max()
)

print("\nNumber of daily records:")
print(len(daily_bed_demand))

Daily bed-demand statistics:
       total_occupied_beds  total_available_beds
count          1096.000000           1096.000000
mean            217.341241            118.114964
std              22.317026             22.322287
min             166.000000             81.000000
25%             198.000000            100.000000
50%             225.000000            110.000000
75%             236.000000            137.250000
max             255.000000            170.000000

Date range:
2023-01-01 00:00:00 to 2025-12-31 00:00:00

Number of daily records:
1096
